# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [27]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('GEMINI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL="gemini-3-flash-preview"
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
openai = OpenAI(base_url= GEMINI_BASE_URL, api_key= api_key)

There might be a problem with your API key? Please visit the troubleshooting notebook!


In [28]:
links = fetch_website_links("https://raspberrypi.tail1b2b1f.ts.net/")
links

['mailto:stawank@zohomail.eu',
 'https://www.linkedin.com/in/stawan-chandrashekhar-kulkarni-810646102/',
 'https://github.com/stawank',
 'tel:+4915785088336',
 'tel:+919823635080',
 'https://github.com/stawank/TU_KL_SeminarElectromobility',
 'https://stawank.wordpress.com/master-thesis/',
 '#',
 'https://github.com/stawank/minimalist-resume']

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [8]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [9]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [10]:
print(get_links_user_prompt("https://raspberrypi.tail1b2b1f.ts.net/"))


Here is the list of links on the website https://raspberrypi.tail1b2b1f.ts.net/ -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

mailto:stawank@zohomail.eu
https://www.linkedin.com/in/stawan-chandrashekhar-kulkarni-810646102/
https://github.com/stawank
tel:+4915785088336
tel:+919823635080
https://github.com/stawank/TU_KL_SeminarElectromobility
https://stawank.wordpress.com/master-thesis/
#
https://github.com/stawank/minimalist-resume


In [29]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [12]:
select_relevant_links("https://raspberrypi.tail1b2b1f.ts.net/")

{'links': [{'type': 'about page',
   'url': 'https://www.linkedin.com/in/stawan-chandrashekhar-kulkarni-810646102/'},
  {'type': 'about page', 'url': 'https://github.com/stawank'},
  {'type': 'about page',
   'url': 'https://stawank.wordpress.com/master-thesis/'}]}

In [13]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [14]:
select_relevant_links("https://raspberrypi.tail1b2b1f.ts.net/")

Selecting relevant links for https://raspberrypi.tail1b2b1f.ts.net/ by calling gemini-3-flash-preview
Found 4 relevant links


{'links': [{'type': 'about page',
   'url': 'https://www.linkedin.com/in/stawan-chandrashekhar-kulkarni-810646102/'},
  {'type': 'about page',
   'url': 'https://stawank.wordpress.com/master-thesis/'},
  {'type': 'portfolio page', 'url': 'https://github.com/stawank'},
  {'type': 'careers page',
   'url': 'https://github.com/stawank/minimalist-resume'}]}

In [15]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3-flash-preview
Found 18 relevant links


{'links': [{'type': 'home page', 'url': 'https://huggingface.co/'},
  {'type': 'about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'organization page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'models page', 'url': 'https://huggingface.co/models'},
  {'type': 'datasets page', 'url': 'https://huggingface.co/datasets'},
  {'type': 'spaces page', 'url': 'https://huggingface.co/spaces'},
  {'type': 'documentation', 'url': 'https://huggingface.co/docs'},
  {'type': 'learning platform', 'url': 'https://huggingface.co/learn'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co'},
  {'type': 'discord community', 'url': 'https://huggingface.co/join/discord'},
  {'type': 'gi

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [16]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [17]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gemini-3-flash-preview
Found 13 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Qwen/Qwen3.5-9B
Updated
5 days ago
•
516k
•
523
Qwen/Qwen3.5-35B-A3B
Updated
7 days ago
•
1M
•
990
Qwen/Qwen3.5-0.8B
Updated
4 days ago
•
265k
•
291
Qwen/Qwen3.5-4B
Updated
5 days ago
•
233k
•
260
unsloth/Qwen3.5-35B-A3B-GGUF
Updated
1 day ago
•
919k
•
547
Browse 2M+ models
Spaces
Running
on
Zero
Featured
376
Omni Video Factory
🏆
376
text to video, image to video, video extend
Running
on
Zero
MCP
1.12k
Wan2.2 14B Preview
🐌
1.12k
generate a video from an image with a text prompt
Running
on
Zero
Featured
1.84k
Qwen Image Mu

In [18]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [19]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [20]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3-flash-preview
Found 11 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nQwen/Qwen3.5-9B\nUpdated\n5 days ago\n•\n516k\n•\n523\nQwen/Qwen3.5-35B-A3B\nUpdated\n7 days ago\n•\n1M\n•\n990\nQwen/Qwen3.5-0.8B\nUpdated\n4 days ago\n•\n265k\n•\n291\nQwen/Qwen3.5-4B\nUpdated\n5 days ago\n•\n233k\n•\n260\nunsloth/Qwen3.5-35B-A3B-GGUF\nUpdated\n1 day ago\n•\n919k\n•\n546\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nFeatured\n376\nOmni Video Factory\n🏆\n376\ntext to vide

In [21]:
get_brochure_user_prompt("Stawank", "https://raspberrypi.tail1b2b1f.ts.net/")

Selecting relevant links for https://raspberrypi.tail1b2b1f.ts.net/ by calling gemini-3-flash-preview
Found 4 relevant links


'\nYou are looking at a company called: Stawank\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nStawan Kulkarni - Automotive Software Engineer\n\n🌙\nEN\n|\nDE\nStawan Chandrashekhar Kulkarni\nAutomotive Software Engineer\n📍 Regensburg, Germany\n📧 stawank@zohomail.eu\n💼 LinkedIn\n🐙 GitHub\n🇩🇪 +49 157 85088336\n🇮🇳 +91 9823635080\n👋 About Me\nPassionate software engineer with 2+ years of experience in\n            automotive software development. I love learning new technologies\n            and now I am learning LLMs in my free time. You will find me either\n            enjoying a good cup of coffee while debugging the latest challenge\n            or binge watching movies.\n💼 Experience\nSystems Engineer\nEDAG Engineering GmbH, Regensburg, Germany\n📅 Feb 2024 - Present\nDeveloping Python-based data fusion and sensor-processing concepts\n          

In [22]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [23]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3-flash-preview
Found 14 relevant links


# Hugging Face: The AI Community Building the Future

Hugging Face is the world’s leading platform for machine learning, serving as the central hub where the global AI community collaborates on models, datasets, and applications. Often referred to as the "home of machine learning," Hugging Face provides the infrastructure and tools necessary to democratize artificial intelligence.

## The Hub of Innovation

Hugging Face offers an unparalleled ecosystem for developers, researchers, and data scientists to build, share, and discover the next generation of AI:

*   **Models:** Explore and download over 2 million models covering every modality, including text, image, video, audio, and 3D.
*   **Datasets:** Access a library of over 500,000 datasets to train and evaluate your machine learning projects.
*   **Spaces:** Browse over 1 million community-built applications. These interactive demos allow users to experience cutting-edge AI—from video generation to 3D camera manipulation—directly in the browser.
*   **Open Source Stack:** Accelerate your development cycle using the industry-standard Hugging Face open-source libraries.

## Solutions for Every Scale

Whether you are an individual hobbyist or a global corporation, Hugging Face provides the tools to succeed.

### For Developers and Teams
*   **Portfolio Building:** Build your professional ML profile by sharing your work with the global community.
*   **Collaboration:** Host unlimited public models and datasets to work seamlessly with others.
*   **Team Plans:** Starting at $20/user/month, teams can access enhanced collaboration features and compute resources.

### For Enterprise
Scale your organization with enterprise-grade security and control. Hugging Face Enterprise provides:
*   **Single Sign-On (SSO):** Securely connect via your existing identity provider.
*   **Data Sovereignty:** Select and audit the specific regions where your repository data is stored.
*   **Audit Logs:** Maintain full visibility with comprehensive logs of all actions taken within the organization.
*   **Dedicated Support:** Access expert guidance to navigate the complexities of AI deployment.

## Company Culture & Vision

At its core, Hugging Face is driven by the spirit of **Open Science and Open Source**. The culture is defined by:

*   **Collaboration over Competition:** By providing a platform for sharing, Hugging Face empowers everyone to build on the shoulders of giants.
*   **Inclusivity in AI:** The company focuses on making AI accessible to everyone, not just those with massive compute budgets.
*   **Rapid Innovation:** With a "trending" ecosystem that updates in real-time, the team and community stay at the absolute forefront of AI research.

## Join the Mission

**For Customers:** Join the thousands of organizations using Hugging Face to accelerate their AI roadmap with the world's most advanced platform.

**For Recruits:** Hugging Face is looking for passionate individuals who want to build the future of technology in the open. Work at the intersection of community and cutting-edge research.

**For Investors:** Hugging Face sits at the center of the AI revolution, acting as the essential infrastructure for the modern machine learning stack and the primary destination for the world's AI talent.

*Visit huggingface.co to explore the future of AI.*

In [24]:
create_brochure("Stawank", "https://raspberrypi.tail1b2b1f.ts.net/" )

Selecting relevant links for https://raspberrypi.tail1b2b1f.ts.net/ by calling gemini-3-flash-preview
Found 5 relevant links


# Stawank: Engineering the Future of Mobility

**Precision. Innovation. Automotive Excellence.**

Stawank is the professional engineering brand of Stawan Kulkarni, a Regensburg-based Automotive Software Engineer specializing in Advanced Driver Assistance Systems (ADAS) and intelligent vehicle software. With a deep foundation in both German and Indian engineering landscapes, Stawank delivers cutting-edge solutions for the next generation of autonomous and electric vehicles.

---

### Core Expertise

Stawank provides high-level technical expertise in the development and validation of complex automotive systems.

*   **ADAS & Sensor Fusion:** Developing Python-based concepts for data fusion and sensor processing to enhance vehicle perception.
*   **Model-Based Development (MBD):** Expert implementation of controller and state logic using MATLAB and Simulink.
*   **Software Validation:** Rigorous unit-test design using TPT, Cmocka, and gTest, ensuring software reliability in safety-critical environments.
*   **Automation & Tooling:** Improving development efficiency through automated test infrastructure and custom Python/C++ tooling.
*   **Electromobility:** Specialized research and development in Energy Management Systems for Hybrid Electric Vehicles (HEVs).

---

### Strategic Experience

Stawank brings a wealth of experience from world-class automotive leaders:

*   **EDAG Engineering GmbH:** Leading systems engineering efforts in data fusion and automated test reporting within agile, international teams.
*   **Continental ADC GmbH:** Prototyping ADAS concepts, automating Ethernet-based camera simulations, and conducting HiL (Hardware-in-the-Loop) testing with OEM components.

---

### Culture and Philosophy

The Stawank approach is defined by a "continuous learning" mindset. Whether it is mastering the latest in Large Language Models (LLMs) or refining C++ interval arithmetic, the focus is always on staying ahead of the technological curve.

*   **Agile Mindset:** Operating within international frameworks to deliver iterative, high-quality results.
*   **Passion for Problem Solving:** A dedication to "debugging the latest challenge" with a blend of technical rigor and creative thinking.
*   **Academic Excellence:** Rooted in advanced technical education from TU Kaiserslautern.

---

### Partnering with Stawank

**For Customers and Investors**
Stawank represents a high-value technical partner for automotive OEMs and Tier-1 suppliers. By leveraging deep domain knowledge in ADAS and automation, Stawank helps partners reduce test effort and simplify complex software architectures.

**For Recruits and Collaborators**
Stawank is always looking to connect with fellow engineers, researchers, and innovators in the automotive space. Collaboration is centered on open-source contributions (GitHub) and pushing the boundaries of what is possible in software-defined vehicles.

---

### Connect with Stawank

Based in the automotive hub of Regensburg, Germany, Stawank is available for consultancy, specialized software projects, and technical partnerships.

*   **Location:** Regensburg, Germany
*   **Email:** stawank@zohomail.eu
*   **GitHub:** stawank
*   **Professional Network:** LinkedIn (Stawan Chandrashekhar Kulkarni)

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [30]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [31]:
stream_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face: The AI Community Building the Future

Hugging Face is the central hub of the machine learning world. As the definitive platform where the global AI community collaborates, we provide the infrastructure and tools necessary to build, discover, and share the next generation of artificial intelligence.

## The Home of Machine Learning

We provide a comprehensive ecosystem for developers, researchers, and organizations to move faster from idea to production. Our platform is built on the philosophy of open-source collaboration, offering:

*   **Models:** Access over 2 million models across every modality, including text, image, video, audio, and 3D.
*   **Datasets:** Browse and utilize over 500,000 datasets to train and evaluate your AI systems.
*   **Spaces:** Explore more than 1 million AI applications and interactive demos hosted directly on our platform.
*   **Open Source Stack:** Leverage the industry-standard Hugging Face tools to accelerate your machine learning workflows.

## Solutions for Every Scale

Whether you are an individual researcher or a global corporation, Hugging Face provides the tools to power your AI journey.

*   **For Individuals:** Build your professional portfolio by sharing your work with the world and engaging with the community’s trending projects.
*   **For Teams & Enterprise:** We offer dedicated Enterprise solutions and paid compute resources to help organizations deploy secure, scalable, and private machine learning environments.
*   **For Researchers:** Stay at the forefront of the field with access to the latest research papers and articles, such as our work on SmolVLM and open-source AI ecosystems.

## Our Culture and Mission

At Hugging Face, we believe that the future of AI should be open, collaborative, and accessible. We are more than just a platform; we are a verified global community of over 84,000 active contributors dedicated to the "open-source way." 

Our culture is rooted in:
*   **Transparency:** Promoting open data and open-source models (like our "Open Data Is All You Need" initiative).
*   **Innovation:** Constantly pushing the boundaries of what is possible in multimodal AI.
*   **Collaboration:** Providing a space where the machine learning community can solve the world's most complex problems together.

## Join the Future of AI

Hugging Face is the place to build your career and your profile in the AI space. By participating in our community, you are joining a global movement to democratize artificial intelligence. 

Explore our trending models, contribute to the latest datasets, or scale your business with our Enterprise solutions. Join the community that is building the future of AI, one model at a time.

In [26]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("Stawank", "https://raspberrypi.tail1b2b1f.ts.net/" )

Selecting relevant links for https://raspberrypi.tail1b2b1f.ts.net/ by calling gemini-3-flash-preview
Found 4 relevant links


# Stawank: Engineering the Future of Automotive Mobility

Based in the automotive hub of Regensburg, Germany, **Stawank** represents the cutting edge of automotive software engineering. Led by Stawan Chandrashekhar Kulkarni, we specialize in developing the intelligent systems that power tomorrow’s vehicles, focusing on Advanced Driver Assistance Systems (ADAS) and robust software architectures.

## Our Expertise

We bridge the gap between complex hardware and intelligent software. Our core competencies include:

*   **ADAS & Sensor Fusion:** Developing Python-based concepts for data fusion and sensor processing to enhance vehicle perception.
*   **Model-Based Development (MBD):** Implementing sophisticated system architectures for vehicle platforms, ensuring reduced time-to-market for new software features.
*   **Automated Testing & Quality:** Expert-level unit testing using TPT, Cmocka, and gTest, combined with automated reporting infrastructure to ensure mission-critical safety.
*   **Full-Stack Automotive Tooling:** Proficiency in C++, Python, MATLAB, and Simulink, alongside specialized tools like CANoe/CAPL for rest-bus simulations.

## Proven Track Record

Our engineering leadership brings a wealth of experience from industry giants and innovative engineering firms, including:

*   **EDAG Engineering GmbH:** Leading systems engineering efforts in international, agile teams.
*   **Continental ADC GmbH:** Prototyping next-generation ADAS architectures and automating Ethernet-based camera driver-assistance tests.
*   **Academic Excellence:** Rooted in research from RPTU Kaiserslautern, with specialized work in Energy Management Systems for Hybrid Electric Vehicles.

## Our Culture

At Stawank, work is driven by curiosity and a passion for technical excellence. We maintain a culture that balances rigorous engineering with a modern, agile mindset:

*   **Continuous Learning:** We don't just keep up with trends; we master them. Current exploration includes the integration of Large Language Models (LLMs) into engineering workflows.
*   **Agile & International:** We thrive in collaborative, multicultural environments that prioritize efficiency and innovation.
*   **The "Human" Element:** Our philosophy is simple—solve the latest debugging challenge with a good cup of coffee, and stay recharged through a shared love for cinema and technology.

## Careers & Collaboration

We are always looking to connect with like-minded innovators, industrial partners, and automotive enthusiasts. Whether you are looking for a technical lead to spearhead a new ADAS function or a partner for model-based system development, Stawank offers the expertise of a seasoned engineer with the agility of a modern developer.

**Join the Journey**
We value technical depth, creative problem-solving, and a proactive approach to the challenges of autonomous driving and electrification.

## Contact Information

**Stawan Chandrashekhar Kulkarni**
*Automotive Software Engineer*

*   **Location:** Regensburg, Germany
*   **Email:** stawank@zohomail.eu
*   **Digital Presence:** [LinkedIn](https://www.linkedin.com) | [GitHub](https://github.com/stawank)
*   **Phone (DE):** +49 157 85088336
*   **Phone (IN):** +91 9823635080

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>